# <b>0. Data Preprocessing.</b>

Build the enriched sub-40 datasets for BTC and ETH.

For each instrument this notebook:
1. Reads raw trades / BBO / liquidations (Binance + Bybit).
2. Splits data into chronological chunks (`chunk_days`).
3. Computes BBO features, markout pnl, rolling VPIN/RV, liquidation windows per chunk.
4. Chronologically sub-samples 1/40 of rows.
5. Concatenates chunks into a single enriched parquet.

The output is written to `data/enriched/{instrument}_sub40.parquet` and consumed by Notebook 1 (meta features) and Notebook 2 (final experiments).

Run this notebook **once per instrument**. Change `INSTRUMENT` and re-execute.

In [ ]:
import sys
import gc
from pathlib import Path
import polars as pl

sys.path.insert(0, str(Path.cwd() / "src"))

from data import DataConfig, build_dataset, count_rows_parquet

## Config

In [2]:
INSTRUMENT = "eth"           # change to "eth" for the second run
SUBSAMPLE_RATE = 40          # keep every 40-th row globally
OVERWRITE = True            # set True to rebuild an existing enriched parquet

# Filenames in the raw data layout:
#   data/binance_booktickers/perp_{instrument}usdt.parquet   (BBO)
#   data/binance_trades/perp_{instrument}usdt.parquet        (trades)
#   data/binance_liquidations/perp_{instrument}usdt.parquet  (Binance liq)
#   data/bybit_liquidations/{instrument}usdt.parquet         (Bybit liq)
DATA_ROOT = Path("data")
OUT_DIR   = Path("data/enriched")
OUT_DIR.mkdir(parents=True, exist_ok=True)

trades_path      = DATA_ROOT / "binance_trades"       / f"perp_{INSTRUMENT}usdt.parquet"
bbo_path         = DATA_ROOT / "binance_booktickers"  / f"perp_{INSTRUMENT}usdt.parquet"
liq_binance_path = DATA_ROOT / "binance_liquidations" / f"perp_{INSTRUMENT}usdt.parquet"
liq_bybit_path   = DATA_ROOT / "bybit_liquidations"   / f"{INSTRUMENT}usdt.parquet"

output_path = OUT_DIR / f"{INSTRUMENT}_sub{SUBSAMPLE_RATE}.parquet"

config = DataConfig(
    chunk_days       = 1,
    windows_s        = (5, 30, 120),
    horizons_s       = (30, 120, 300),
    rebate_bps       = 0.5,
    clipped_notional = 100_000.0,
    bbo_lookback_s   = 120,
    bybit_shift_ms   = 200,
    n_min_obs        = 3
)

print(f"Instrument       : {INSTRUMENT}")
print(f"Subsample rate   : {SUBSAMPLE_RATE}")
print(f"Trades           : {trades_path}")
print(f"BBO              : {bbo_path}")
print(f"Liq binance      : {liq_binance_path}")
print(f"Liq bybit        : {liq_bybit_path}")
print(f"Output           : {output_path}")
print(f"Windows (rolling): {config.windows_s} s")
print(f"Horizons (pnl)   : {config.horizons_s} s")

Instrument       : eth
Subsample rate   : 40
Trades           : data\binance_trades\perp_ethusdt.parquet
BBO              : data\binance_booktickers\perp_ethusdt.parquet
Liq binance      : data\binance_liquidations\perp_ethusdt.parquet
Liq bybit        : data\bybit_liquidations\ethusdt.parquet
Output           : data\enriched\eth_sub40.parquet
Windows (rolling): (5, 30, 120) s
Horizons (pnl)   : (30, 120, 300) s


## Build

In [3]:
build_dataset(
    trades_path       = trades_path,
    bbo_path          = bbo_path,
    liq_binance_path  = liq_binance_path,
    liq_bybit_path    = liq_bybit_path,
    output_path       = output_path,
    config            = config,
    subsample_rate    = SUBSAMPLE_RATE,
    overwrite         = OVERWRITE,
    keep_chunk_files  = False,
    align_to_utc      = True,
)

gc.collect()

Building enriched dataset:   0%|          | 0/179 [00:00<?, ?it/s]

0

## Validation

In [4]:
# Quick sanity check on the written parquet.
n_rows = count_rows_parquet(output_path)
size_mb = output_path.stat().st_size / 1024**2

print(f"File     : {output_path.name}")
print(f"Rows     : {n_rows:,}")
print(f"Size     : {size_mb:,.1f} MB")

df_head = pl.read_parquet(output_path, n_rows=5)
print(f"\nColumns  ({len(df_head.columns)}):")
for c in df_head.columns:
    print(f"  {c}")

File     : eth_sub40.parquet
Rows     : 34,261,282
Size     : 1,546.1 MB

Columns  (25):
  timestamp
  price
  mid_price
  spread
  obi
  dist_to_mid
  sign
  notional
  weight
  hour
  vpin_5s
  vpin_30s
  vpin_120s
  rv_5s
  rv_30s
  rv_120s
  liq_binance_5s
  liq_bybit_5s
  liq_binance_30s
  liq_bybit_30s
  liq_binance_120s
  liq_bybit_120s
  pnl_30s
  pnl_120s
  pnl_300s


In [5]:
# Coverage across walk-forward windows.
from splits import WINDOWS

df_ts = pl.read_parquet(output_path, columns=["timestamp"])
for wid, spec in WINDOWS.items():
    n_train = df_ts.filter(
        (pl.col("timestamp") >= spec.train_start) & (pl.col("timestamp") < spec.train_end)
    ).height
    n_test = df_ts.filter(
        (pl.col("timestamp") >= spec.embargo_end) & (pl.col("timestamp") < spec.test_end)
    ).height
    print(f"{wid}: train={n_train:>12,}  test={n_test:>12,}")

del df_ts
gc.collect()

W1: train=   6,674,677  test=   3,803,064
W2: train=  11,799,050  test=   3,784,745
W3: train=  16,399,206  test=   4,327,318
W4: train=  24,321,686  test=   4,407,417
W5: train=  30,508,564  test=   2,819,501


0